# Advanced 95-Feature Pipeline: ElasticNet & Ridge Logistic Regression

Multi-target intraoperative prediction across **95 Statistical Features** with **STRIDE = 10**:
- Models: Regularized `LogisticRegression` (L1, L2, ElasticNet) and `CalibratedClassifierCV` linear probability models.
- Features: CPU-tuned regularization penalty ($C$, $\ell_1$-ratio) with feature weight extraction for clinical biomarker interpretability.

In [ ]:
import os
import gc
import glob
import json
import random
import warnings
import time
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import njit
from joblib import Parallel, delayed

from sklearn.model_selection import train_test_split, PredefinedSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay
)
import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Advanced ML Environment, Scikit-Learn & CPU Parallel Acceleration Loaded Successfully.")

In [ ]:
# ======================================================
# High-Performance Dataset Loading & 95-Feature Rolling Window Engine
# ======================================================

USE_FULL_DATASET = True
CACHE_DATASET = True
ENABLE_TUNING = True
WINDOW_SIZE = 600
STRIDE = 10
N_JOBS = -1
MAX_TRAIN_PATIENTS = None  # Set to integer (e.g. 500) if testing on memory-restricted machines

base_dir = os.getcwd()
candidates = [
    os.path.join(base_dir, "patient_labeled_data"),
    os.path.join(base_dir, "..", "patient_labeled_data"),
    os.path.join(base_dir, "..", "..", "patient_labeled_data")
]
input_dir = next((c for c in candidates if os.path.exists(c)), candidates[1])
csv_files = sorted(glob.glob(os.path.join(input_dir, "patient_*_1hz.csv")))

base_features = [
    "Solar8000/HR",
    "Solar8000/ART_SBP",
    "Solar8000/ART_DBP",
    "Solar8000/ART_MBP",
    "Solar8000/PLETH_SPO2",
    "Solar8000/RR_CO2",
    "Solar8000/ETCO2",
    "Primus/FIO2",
    "Solar8000/BT"
]

engineered_features = [
    "Feature_Pulse_Pressure",
    "Feature_Shock_Index",
    "Feature_Modified_Shock_Index",
    "Feature_Rate_Pressure_Product",
    "Feature_HR_Mean_60s",
    "Feature_HR_Std_60s",
    "Feature_HR_Delta_60s",
    "Feature_MBP_Mean_60s",
    "Feature_MBP_Std_60s",
    "Feature_MBP_Delta_60s"
]

features_19 = base_features + engineered_features
clean_names_19 = [col.replace("Solar8000/", "").replace("Primus/", "") for col in features_19]
stat_names = ["mean", "std", "min", "max", "slope"]
window_feature_names = [f"{col}_{stat}" for stat in stat_names for col in clean_names_19]

target_cols = ["Future_Hypotension", "Future_Hypoxia", "Future_Tachycardia"]
all_req_cols = features_19 + target_cols

train_val_files, test_files = train_test_split(csv_files, test_size=0.20, random_state=42, shuffle=True)
train_files, val_files = train_test_split(train_val_files, test_size=0.125, random_state=42, shuffle=True)

if not USE_FULL_DATASET:
    train_subset = train_files[:300]
    val_subset = val_files[:50]
    test_subset = test_files[:100]
elif MAX_TRAIN_PATIENTS is not None:
    train_subset = train_files[:MAX_TRAIN_PATIENTS]
    val_subset = val_files
    test_subset = test_files
else:
    train_subset = train_files
    val_subset = val_files
    test_subset = test_files

print(f"Dataset Path : {input_dir}")
print(f"Total Patients: {len(csv_files)} | Training: {len(train_subset)} | Val: {len(val_subset)} | Test: {len(test_subset)}")
print(f"Feature Count : {len(window_feature_names)} Statistical Window Features (W={WINDOW_SIZE}s, STRIDE={STRIDE}s)")

# Numba Fast Vectorized Window Extractor (Stat-Grouped to match StandardScaler & EFR32 firmware layout)
@njit(fastmath=True)
def extract_windows_95_all_targets(arr, y_arr, window_size, stride):
    n_rows, n_cols = arr.shape
    n_windows = (n_rows - window_size) // stride
    if n_windows <= 0:
        return np.empty((0, n_cols * 5), dtype=np.float32), np.empty((0, y_arr.shape[1]), dtype=np.float32)
    
    out_X = np.empty((n_windows, n_cols * 5), dtype=np.float32)
    out_y = np.empty((n_windows, y_arr.shape[1]), dtype=np.float32)
    inv_w = 1.0 / window_size
    
    for w in range(n_windows):
        start = w * stride
        end = start + window_size
        
        for col in range(n_cols):
            val_first = arr[start, col]
            val_last = arr[end - 1, col]
            s = 0.0
            sq_s = 0.0
            mn = arr[start, col]
            mx = arr[start, col]
            
            for i in range(start, end):
                v = arr[i, col]
                s += v
                sq_s += v * v
                if v < mn: mn = v
                if v > mx: mx = v
                
            mean = s * inv_w
            var = (sq_s * inv_w) - (mean * mean)
            std = np.sqrt(max(0.0, var))
            slope = (val_last - val_first) / (window_size + 1e-5)
            
            # 5 Stat Groups:
            out_X[w, 0 * n_cols + col] = mean
            out_X[w, 1 * n_cols + col] = std
            out_X[w, 2 * n_cols + col] = mn
            out_X[w, 3 * n_cols + col] = mx
            out_X[w, 4 * n_cols + col] = slope
            
        for t in range(y_arr.shape[1]):
            out_y[w, t] = y_arr[end - 1, t]
            
    return out_X, out_y

# JIT Warmup
_ = extract_windows_95_all_targets(np.zeros((700, 19), dtype=np.float32), np.zeros((700, 3), dtype=np.float32), 600, 10)

def _process_single_csv(file_path):
    try:
        df = pd.read_csv(file_path, usecols=all_req_cols, dtype=np.float32, engine="c")
        if df.empty or len(df) <= WINDOW_SIZE:
            return None
        df = df.ffill().bfill().fillna(0)
        arr = df[features_19].to_numpy(dtype=np.float32)
        y_arr = df[target_cols].to_numpy(dtype=np.float32)
        return extract_windows_95_all_targets(arr, y_arr, WINDOW_SIZE, STRIDE)
    except Exception:
        return None

def build_split_parallel(file_list, n_jobs=N_JOBS):
    results = Parallel(n_jobs=n_jobs, prefer="threads")(delayed(_process_single_csv)(f) for f in file_list)
    results = [r for r in results if r is not None and len(r[0]) > 0]
    if not results:
        return np.empty((0, 95), dtype=np.float32), np.empty((0, 3), dtype=np.float32)
    X = np.concatenate([r[0] for r in results], axis=0)
    y = np.concatenate([r[1] for r in results], axis=0)
    return X, y

# Cache management for sub-second loading
cache_dir = os.path.join(base_dir, "cache")
os.makedirs(cache_dir, exist_ok=True)
cache_file = os.path.join(cache_dir, f"dataset_95_w{WINDOW_SIZE}_s{STRIDE}_{len(train_subset)}.npz")

if CACHE_DATASET and os.path.exists(cache_file):
    print(f"[Dataset Cache Hit] Loading precomputed splits from {cache_file}...")
    t0 = time.time()
    data = np.load(cache_file)
    X_tr, y_tr_all = data["X_tr"], data["y_tr"]
    X_va, y_va_all = data["X_va"], data["y_va"]
    X_te, y_te_all = data["X_te"], data["y_te"]
    print(f"Loaded from cache in {time.time() - t0:.2f}s!")
else:
    print(f"[Dataset Extraction] Extracting 95 features in parallel across {os.cpu_count()} CPU cores...")
    t0 = time.time()
    X_tr, y_tr_all = build_split_parallel(train_subset)
    X_va, y_va_all = build_split_parallel(val_subset)
    X_te, y_te_all = build_split_parallel(test_subset)
    print(f"Feature Extraction Completed in {time.time() - t0:.2f}s!")
    if CACHE_DATASET:
        np.savez_compressed(cache_file, X_tr=X_tr, y_tr=y_tr_all, X_va=X_va, y_va=y_va_all, X_te=X_te, y_te=y_te_all)
        print(f"Saved dataset cache to {cache_file}")

print(f"Matrix Shapes -> Train: X={X_tr.shape}, y={y_tr_all.shape} | Val: X={X_va.shape}, y={y_va_all.shape} | Test: X={X_te.shape}, y={y_te_all.shape}")
print(f"RAM Footprint -> Train X: {X_tr.nbytes / (1024**2):.1f} MB (float32)")

# Helper: Scaler loading & caching
def save_scaler_params_to_json(scaler, feature_names_list, json_path):
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    scaler_dict = OrderedDict([
        ("mean", OrderedDict((feat, float(scaler.mean_[i])) for i, feat in enumerate(feature_names_list))),
        ("variance", OrderedDict((feat, float(scaler.var_[i])) for i, feat in enumerate(feature_names_list))),
        ("std", OrderedDict((feat, float(scaler.scale_[i])) for i, feat in enumerate(feature_names_list)))
    ])
    with open(json_path, "w") as f:
        json.dump(scaler_dict, f, indent=4)
    print(f"[Scaler Export] Saved Scaler parameters to JSON: {json_path}")
    return scaler_dict

def get_or_fit_scaler(json_path, feature_names_list, X_train=None, force_recompute=False):
    scaler = StandardScaler()
    if not force_recompute and os.path.exists(json_path):
        try:
            with open(json_path, "r") as f:
                sc_data = json.load(f)
            if "mean" in sc_data and "std" in sc_data:
                mean_vals = [sc_data["mean"][k] for k in feature_names_list if k in sc_data["mean"]]
                scale_vals = [sc_data["std"][k] for k in feature_names_list if k in sc_data["std"]]
                var_vals = [sc_data.get("variance", {}).get(k, sc_data["std"][k]**2) for k in feature_names_list if k in sc_data.get("variance", sc_data["std"])]
                if len(mean_vals) == len(feature_names_list) and len(scale_vals) == len(feature_names_list):
                    scaler.mean_ = np.array(mean_vals, dtype=np.float64)
                    scaler.scale_ = np.array(scale_vals, dtype=np.float64)
                    scaler.var_ = np.array(var_vals, dtype=np.float64)
                    scaler.n_features_in_ = len(feature_names_list)
                    print(f"[Scaler Cache Hit] Loaded existing StandardScaler from: {json_path}")
                    return scaler
        except Exception as e:
            print(f"[Scaler Warning] Failed to load {json_path} ({e}). Refitting...")
            
    print(f"[Scaler Compute] Fitting StandardScaler from training data...")
    if X_train is not None and len(X_train) > 0:
        scaler.fit(X_train)
        save_scaler_params_to_json(scaler, feature_names_list, json_path)
    return scaler

# Helper: Metrics & Threshold Tuning
def get_optimal_tau(y_t, y_p):
    best_tau, best_f1 = 0.5, -1.0
    for tau in np.linspace(0.01, 0.99, 99):
        f1 = f1_score(y_t, (y_p >= tau).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_tau = f1, tau
    return best_tau

def compute_metrics(y_t, y_p, tau):
    y_b = (y_p >= tau).astype(int)
    auroc = roc_auc_score(y_t, y_p)
    auprc = average_precision_score(y_t, y_p)
    acc = accuracy_score(y_t, y_b)
    bal_acc = balanced_accuracy_score(y_t, y_b)
    prec = precision_score(y_t, y_b, zero_division=0)
    rec = recall_score(y_t, y_b, zero_division=0)
    f1 = f1_score(y_t, y_b, zero_division=0)
    mcc = matthews_corrcoef(y_t, y_b)
    tn, fp, fn, tp = confusion_matrix(y_t, y_b).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return dict(auroc=auroc, auprc=auprc, acc=acc, bal_acc=bal_acc, prec=prec, rec=rec, spec=spec, f1=f1, mcc=mcc, tn=tn, fp=fp, fn=fn, tp=tp)

def print_metrics_table(target_name, model_name, y_te, test_probs, optimal_tau):
    m_def = compute_metrics(y_te, test_probs, 0.50)
    m_opt = compute_metrics(y_te, test_probs, optimal_tau)
    print("=" * 76)
    print(f"  TEST METRICS: {model_name} | TARGET: {target_name}")
    print("=" * 76)
    print(f"Metric                 Default (tau=0.50)       OPTIMAL (tau*={optimal_tau:.2f})")
    print("-" * 76)
    print(f"AUROC (ROC AUC)         : {m_def['auroc']:.4f}                  {m_opt['auroc']:.4f}")
    print(f"AUPRC (PR AUC)          : {m_def['auprc']:.4f}                  {m_opt['auprc']:.4f}")
    print(f"Accuracy                : {m_def['acc']:.4f}                  {m_opt['acc']:.4f}")
    print(f"Balanced Accuracy       : {m_def['bal_acc']:.4f}                  {m_opt['bal_acc']:.4f}")
    print(f"Sensitivity / Recall    : {m_def['rec']:.4f}                  {m_opt['rec']:.4f}")
    print(f"Specificity (TNR)       : {m_def['spec']:.4f}                  {m_opt['spec']:.4f}")
    print(f"Precision (PPV)         : {m_def['prec']:.4f}                  {m_opt['prec']:.4f}")
    print(f"F1 Score                : {m_def['f1']:.4f}                  {m_opt['f1']:.4f}")
    print(f"MCC                     : {m_def['mcc']:.4f}                  {m_opt['mcc']:.4f}")
    print("-" * 76)
    print(f"Confusion Matrix (0.50) : TN={m_def['tn']}, FP={m_def['fp']}, FN={m_def['fn']}, TP={m_def['tp']}")
    print(f"Confusion Matrix (tau*) : TN={m_opt['tn']}, FP={m_opt['fp']}, FN={m_opt['fn']}, TP={m_opt['tp']}")
    print("=" * 76)
    print()
    return m_opt

def plot_evaluation_charts(target_name, y_te, test_probs, optimal_tau):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fpr, tpr, _ = roc_curve(y_te, test_probs)
    prec_pts, rec_pts, _ = precision_recall_curve(y_te, test_probs)
    auc_val = roc_auc_score(y_te, test_probs)
    auprc_val = average_precision_score(y_te, test_probs)
    
    axes[0].plot(fpr, tpr, label=f"ROC (AUC = {auc_val:.3f})", color="darkorange", lw=2)
    axes[0].plot(rec_pts, prec_pts, label=f"PR (AUC = {auprc_val:.3f})", color="purple", lw=2)
    axes[0].plot([0, 1], [0, 1], color="gray", linestyle=":")
    axes[0].set_title(f"ROC & PR Curves: {target_name}")
    axes[0].set_xlabel("FPR / Recall")
    axes[0].set_ylabel("TPR / Precision")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    cm_d1 = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs >= 0.50).astype(int)), display_labels=["Neg", "Pos"])
    cm_d1.plot(ax=axes[1], cmap="Reds", colorbar=False)
    axes[1].set_title("CM at tau=0.50")

    cm_d2 = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_te, (test_probs >= optimal_tau).astype(int)), display_labels=["Neg", "Pos"])
    cm_d2.plot(ax=axes[2], cmap="Greens", colorbar=False)
    axes[2].set_title(f"CM at Optimal tau*={optimal_tau:.2f}")

    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================================================================
# Target: Future_Hypotension (Target Index: 0)
# ==============================================================================
target_idx = 0
target_name = target_cols[target_idx]
print(f"=== [TRAINING & CPU TUNING] Logistic Regression for {target_name} ===")

y_tr = y_tr_all[:, target_idx]
y_va = y_va_all[:, target_idx]
y_te = y_te_all[:, target_idx]

scaler_path = os.path.join("models", "scalers", f"scaler_{target_name}.json")
scaler = get_or_fit_scaler(scaler_path, window_feature_names, X_tr)

X_tr_sc = scaler.transform(X_tr)
X_va_sc = scaler.transform(X_va)
X_te_sc = scaler.transform(X_te)

X_comb = np.vstack([X_tr_sc, X_va_sc])
y_comb = np.concatenate([y_tr, y_va])
split_indices = np.concatenate([-1 * np.ones(len(X_tr), dtype=int), np.zeros(len(X_va), dtype=int)])
pds = PredefinedSplit(test_fold=split_indices)

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV

if ENABLE_TUNING:
    lr_grid = {
        "C": [0.01, 0.1, 1.0, 10.0],
        "penalty": ["l2"],
        "class_weight": ["balanced", None]
    }
    lr_search = RandomizedSearchCV(
        LogisticRegression(solver="lbfgs", max_iter=300, random_state=42),
        lr_grid,
        n_iter=6,
        scoring="roc_auc",
        cv=pds,
        n_jobs=-1,
        random_state=42
    )
    lr_search.fit(X_comb, y_comb)
    lr = lr_search.best_estimator_
    print(f"[LR Tuning] Best Params: {lr_search.best_params_} | Validation AUROC: {lr_search.best_score_:.4f}")
else:
    lr = LogisticRegression(C=0.1, penalty="l2", class_weight="balanced", solver="lbfgs", max_iter=300, random_state=42)
    lr.fit(X_tr_sc, y_tr)

lr_val_probs = lr.predict_proba(X_va_sc)[:, 1]
tau_lr = get_optimal_tau(y_va, lr_val_probs)
lr_test_probs = lr.predict_proba(X_te_sc)[:, 1]
print_metrics_table(target_name, "Logistic Regression (95 feats)", y_te, lr_test_probs, tau_lr)

lr_path = os.path.join("models", "logistic_regression", f"lr_{target_name}.joblib")
os.makedirs(os.path.dirname(lr_path), exist_ok=True)
joblib.dump(lr, lr_path)
print(f"Saved Logistic Regression model to: {lr_path}")

# Top feature importance
coefs = lr.coef_[0]
top_pos_idx = np.argsort(coefs)[-5:][::-1]
top_neg_idx = np.argsort(coefs)[:5]
print(f"Top 5 Positive Risk Predictors: {[window_feature_names[i] for i in top_pos_idx]}")
print(f"Top 5 Negative/Protective Predictors: {[window_feature_names[i] for i in top_neg_idx]}")

plot_evaluation_charts(f"{target_name} (Logistic Regression 95 Feats)", y_te, lr_test_probs, tau_lr)

del X_comb, y_comb, X_tr_sc, X_va_sc, X_te_sc
gc.collect()

In [ ]:
# ==============================================================================
# Target: Future_Hypoxia (Target Index: 1)
# ==============================================================================
target_idx = 1
target_name = target_cols[target_idx]
print(f"=== [TRAINING & CPU TUNING] Logistic Regression for {target_name} ===")

y_tr = y_tr_all[:, target_idx]
y_va = y_va_all[:, target_idx]
y_te = y_te_all[:, target_idx]

scaler_path = os.path.join("models", "scalers", f"scaler_{target_name}.json")
scaler = get_or_fit_scaler(scaler_path, window_feature_names, X_tr)

X_tr_sc = scaler.transform(X_tr)
X_va_sc = scaler.transform(X_va)
X_te_sc = scaler.transform(X_te)

X_comb = np.vstack([X_tr_sc, X_va_sc])
y_comb = np.concatenate([y_tr, y_va])
split_indices = np.concatenate([-1 * np.ones(len(X_tr), dtype=int), np.zeros(len(X_va), dtype=int)])
pds = PredefinedSplit(test_fold=split_indices)

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV

if ENABLE_TUNING:
    lr_grid = {
        "C": [0.01, 0.1, 1.0, 10.0],
        "penalty": ["l2"],
        "class_weight": ["balanced", None]
    }
    lr_search = RandomizedSearchCV(
        LogisticRegression(solver="lbfgs", max_iter=300, random_state=42),
        lr_grid,
        n_iter=6,
        scoring="roc_auc",
        cv=pds,
        n_jobs=-1,
        random_state=42
    )
    lr_search.fit(X_comb, y_comb)
    lr = lr_search.best_estimator_
    print(f"[LR Tuning] Best Params: {lr_search.best_params_} | Validation AUROC: {lr_search.best_score_:.4f}")
else:
    lr = LogisticRegression(C=0.1, penalty="l2", class_weight="balanced", solver="lbfgs", max_iter=300, random_state=42)
    lr.fit(X_tr_sc, y_tr)

lr_val_probs = lr.predict_proba(X_va_sc)[:, 1]
tau_lr = get_optimal_tau(y_va, lr_val_probs)
lr_test_probs = lr.predict_proba(X_te_sc)[:, 1]
print_metrics_table(target_name, "Logistic Regression (95 feats)", y_te, lr_test_probs, tau_lr)

lr_path = os.path.join("models", "logistic_regression", f"lr_{target_name}.joblib")
os.makedirs(os.path.dirname(lr_path), exist_ok=True)
joblib.dump(lr, lr_path)
print(f"Saved Logistic Regression model to: {lr_path}")

# Top feature importance
coefs = lr.coef_[0]
top_pos_idx = np.argsort(coefs)[-5:][::-1]
top_neg_idx = np.argsort(coefs)[:5]
print(f"Top 5 Positive Risk Predictors: {[window_feature_names[i] for i in top_pos_idx]}")
print(f"Top 5 Negative/Protective Predictors: {[window_feature_names[i] for i in top_neg_idx]}")

plot_evaluation_charts(f"{target_name} (Logistic Regression 95 Feats)", y_te, lr_test_probs, tau_lr)

del X_comb, y_comb, X_tr_sc, X_va_sc, X_te_sc
gc.collect()

In [ ]:
# ==============================================================================
# Target: Future_Tachycardia (Target Index: 2)
# ==============================================================================
target_idx = 2
target_name = target_cols[target_idx]
print(f"=== [TRAINING & CPU TUNING] Logistic Regression for {target_name} ===")

y_tr = y_tr_all[:, target_idx]
y_va = y_va_all[:, target_idx]
y_te = y_te_all[:, target_idx]

scaler_path = os.path.join("models", "scalers", f"scaler_{target_name}.json")
scaler = get_or_fit_scaler(scaler_path, window_feature_names, X_tr)

X_tr_sc = scaler.transform(X_tr)
X_va_sc = scaler.transform(X_va)
X_te_sc = scaler.transform(X_te)

X_comb = np.vstack([X_tr_sc, X_va_sc])
y_comb = np.concatenate([y_tr, y_va])
split_indices = np.concatenate([-1 * np.ones(len(X_tr), dtype=int), np.zeros(len(X_va), dtype=int)])
pds = PredefinedSplit(test_fold=split_indices)

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV

if ENABLE_TUNING:
    lr_grid = {
        "C": [0.01, 0.1, 1.0, 10.0],
        "penalty": ["l2"],
        "class_weight": ["balanced", None]
    }
    lr_search = RandomizedSearchCV(
        LogisticRegression(solver="lbfgs", max_iter=300, random_state=42),
        lr_grid,
        n_iter=6,
        scoring="roc_auc",
        cv=pds,
        n_jobs=-1,
        random_state=42
    )
    lr_search.fit(X_comb, y_comb)
    lr = lr_search.best_estimator_
    print(f"[LR Tuning] Best Params: {lr_search.best_params_} | Validation AUROC: {lr_search.best_score_:.4f}")
else:
    lr = LogisticRegression(C=0.1, penalty="l2", class_weight="balanced", solver="lbfgs", max_iter=300, random_state=42)
    lr.fit(X_tr_sc, y_tr)

lr_val_probs = lr.predict_proba(X_va_sc)[:, 1]
tau_lr = get_optimal_tau(y_va, lr_val_probs)
lr_test_probs = lr.predict_proba(X_te_sc)[:, 1]
print_metrics_table(target_name, "Logistic Regression (95 feats)", y_te, lr_test_probs, tau_lr)

lr_path = os.path.join("models", "logistic_regression", f"lr_{target_name}.joblib")
os.makedirs(os.path.dirname(lr_path), exist_ok=True)
joblib.dump(lr, lr_path)
print(f"Saved Logistic Regression model to: {lr_path}")

# Top feature importance
coefs = lr.coef_[0]
top_pos_idx = np.argsort(coefs)[-5:][::-1]
top_neg_idx = np.argsort(coefs)[:5]
print(f"Top 5 Positive Risk Predictors: {[window_feature_names[i] for i in top_pos_idx]}")
print(f"Top 5 Negative/Protective Predictors: {[window_feature_names[i] for i in top_neg_idx]}")

plot_evaluation_charts(f"{target_name} (Logistic Regression 95 Feats)", y_te, lr_test_probs, tau_lr)

del X_comb, y_comb, X_tr_sc, X_va_sc, X_te_sc
gc.collect()